# Programmations parallèle & concurrente

## Appeler parallélement une fonction avec un argument

Appelez la fonction `f` qui donne des infos sur le process qui l'exécute parallélement avec une `multiprocessing.Pool`.

In [ ]:
import os


def f(n: int) -> None:
  print(f"Process {n}")
  print(f"ID du process parent : {os.getppid()}")
  print(f"ID du process : {os.getpid()}")


# Votre code ici

### Solution

In [ ]:
import multiprocessing
import os


def f(n: int) -> None:
  print(f"Process {n}")
  print(f"ID du process parent : {os.getppid()}")
  print(f"ID du process : {os.getpid()}")


with multiprocessing.Pool() as pool:
  pool.map(f, range(10))

## Appeler en parallèle une fonction dont un ou plusieurs arguments sont constants

Utilisez [`functools.partial`](https://docs.python.org/fr/3/library/functools.html#functools.partial) et adaptez le code précédent pour appeler `f` avec des `n` allant de 0 à 9 et `verbose` toujours fixé à `False`.

In [ ]:
import os


def f(n: int, verbose: bool) -> int:
  if verbose:
    print(f"Process {n}")
    print(f"ID du process parent : {os.getppid()}")
    print(f"ID du process : {os.getpid()}")
  return n * 2


# Votre code ici

### Solution

In [ ]:
import functools
import multiprocessing
import os


def f(n: int, verbose: bool) -> None:
  if verbose:
    print(f"Process {n}")
    print(f"ID du process parent : {os.getppid()}")
    print(f"ID du process : {os.getpid()}")
  return n * 2


with multiprocessing.Pool() as pool:
  ns = pool.map(functools.partial(f, verbose=False), range(10))

print(ns)

## Télécharger plusieurs fichiers simultanément, avec un seul argument

Utilisez un `multiprocessing.pool.ThreadPool` pour télécharger plusieurs fichiers simultanément. Dans un premier temps, nous allons télécharger 10 fois la page aléatoire de Wikipédia : `https://en.wikipedia.org/wiki/Special:Random`. La fonction parallélisée récupèrera simplement un entier et stockera le résultat du téléchargement dans un dossier `articles`, sous `{n}.html` si `n` est l'entier.

Pour le téléchargement de fichier, vous pourrez utiliser le code suivant :

```python
with urllib.request.urlopen(url) as response, path.open("wb") as fh:
    shutil.copyfileobj(response, fh)
```

où `url` est l'URL à télécharger et `path` est le chemin où écrire le fichier.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import multiprocessing.pool
import pathlib
import shutil
from urllib.request import Request, urlopen
import urllib.request


articles = pathlib.Path("articles")
articles.mkdir(exist_ok=True)
url = "https://en.wikipedia.org/wiki/Special:Random"


def download_article(n: int):
  request = Request(url)
  request.add_header(
    "User-Agent",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.11 (KHTML, like Gecko) Chrome/23.0.1271.64 Safari/537.11",
  )
  request.add_header(
    "Accept", "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
  )
  request.add_header("Accept-Charset", "ISO-8859-1,utf-8;q=0.7,*;q=0.3")
  request.add_header("Accept-Encoding", "none")
  request.add_header("Accept-Language", "en-US,en;q=0.8")
  request.add_header("Connection", "keep-alive")
  with (
    urllib.request.urlopen(request) as response,
    (articles / f"{n}.html").open("wb") as fh,
  ):
    shutil.copyfileobj(response, fh)


with multiprocessing.pool.ThreadPool() as pool:
  results = pool.map(download_article, range(10))

## Télécharger plusieurs fichiers simultanément, avec deux arguments

De manière similaire à l'exercice précédent, on souhaite télécharger plusieurs pages en même temps. Cette fois on souhaite donner à notre worker une url et un chemin où écrire, plutôt que seulement un entier.

Adaptez le code précédent pour télécharger `to_download`.

In [ ]:
import pathlib


downloads = pathlib.Path("downloads")
downloads.mkdir(exist_ok=True)

to_download = (
  ("https://docs.python.org/fr/3/", downloads / "python-docs.html"),
  ("http://pythontutor.com/", downloads / "python-tutor.html"),
  ("https://www.google.com/", downloads / "google.html"),
)

# Votre code ici

### Solution

In [ ]:
import multiprocessing.pool
import pathlib
import shutil
import time
import typing
import urllib.request


downloads = pathlib.Path("downloads")
downloads.mkdir(exist_ok=True)

to_download = (
  ("https://docs.python.org/", downloads / "python-docs.html"),
  ("http://pythontutor.com/", downloads / "python-tutor.html"),
  ("https://www.google.com/", downloads / "google.html"),
)


def download(url: str, path: pathlib.Path) -> None:
  with urllib.request.urlopen(url) as response, path.open("wb") as fh:
    shutil.copyfileobj(response, fh)


with multiprocessing.pool.ThreadPool() as pool:
  pool.starmap(download, to_download)

## Créer une architecture producteur / consommateur

In [ ]:
import multiprocessing


def consumer(queue: multiprocessing.Queue) -> None:
  while True:
    item = queue.get()
    if item is None:
      break
    print(item)


def producer(queue: multiprocessing.Queue) -> None:
  for i in range(10):
    queue.put(i)
  queue.put(None)


if __name__ == "__main__":
  queue = multiprocessing.Queue()
  consumer = multiprocessing.Process(target=consumer, args=(queue,))
  producer = multiprocessing.Process(target=producer, args=(queue,))
  consumer.start()
  producer.start()
  consumer.join()
  producer.join()

## Effectuer deux tâches différentes en parallèle avec un `Pool`

In [ ]:
import multiprocessing


def a(i: int) -> int:
  return i * 2


def b(i: int) -> int:
  return i**2


if __name__ == "__main__":
  with multiprocessing.Pool() as pool:
    a_results = pool.map_async(a, range(10))
    b_results = pool.map_async(b, range(10))
    print(a_results.get())
    print(b_results.get())